In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, numpy as np

EMB_DIR = '/content/drive/MyDrive/Colab Notebooks/embeddings_256'  # your path to her biolinkbert folder
# List recursively to see the full structure
for root, dirs, files in os.walk(f'{EMB_DIR}/biolinkbert'):
    rel = root.replace(EMB_DIR, '')
    print(f"\n{rel}/")
    for d in sorted(dirs):
        print(f"  [DIR] {d}")
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024**2
        print(f"  {f}  ({size:.1f} MB)")


/biolinkbert/
  [DIR] gpt4o
  [DIR] hybrid
  [DIR] tier1
  [DIR] tier2

/biolinkbert/tier1/
  [DIR] nonlinear_ae

/biolinkbert/tier1/nonlinear_ae/
  ae_decoder.pt  (1.0 MB)
  ae_encoder.pt  (1.0 MB)
  ae_meta.json  (0.0 MB)
  drug_embeddings.pt  (11.2 MB)
  phenotype_embeddings.pt  (11.2 MB)
  projection_fit_info.json  (0.0 MB)

/biolinkbert/tier2/
  [DIR] nonlinear_ae

/biolinkbert/tier2/nonlinear_ae/
  ae_decoder.pt  (1.0 MB)
  ae_encoder.pt  (1.0 MB)
  ae_meta.json  (0.0 MB)
  drug_embeddings.pt  (11.2 MB)
  phenotype_embeddings.pt  (11.2 MB)
  projection_fit_info.json  (0.0 MB)

/biolinkbert/gpt4o/
  [DIR] nonlinear_ae

/biolinkbert/gpt4o/nonlinear_ae/
  ae_decoder.pt  (1.0 MB)
  ae_encoder.pt  (1.0 MB)
  ae_meta.json  (0.0 MB)
  drug_embeddings.pt  (11.2 MB)
  phenotype_embeddings.pt  (11.2 MB)
  projection_fit_info.json  (0.0 MB)

/biolinkbert/hybrid/
  [DIR] nonlinear_ae

/biolinkbert/hybrid/nonlinear_ae/
  ae_decoder.pt  (1.0 MB)
  ae_encoder.pt  (1.0 MB)
  ae_meta.json  (0.0 

In [ ]:
import json, torch

TIER = 'hybrid'  # start with this one
NLA_DIR = f'{EMB_DIR}/biolinkbert/{TIER}/nonlinear_ae'

# Read metadata
with open(f'{NLA_DIR}/ae_meta.json') as f:
    meta = json.load(f)
print("ae_meta.json:")
print(json.dumps(meta, indent=2))

with open(f'{NLA_DIR}/projection_fit_info.json') as f:
    proj = json.load(f)
print("\nprojection_fit_info.json:")
print(json.dumps(proj, indent=2))

# Inspect the embeddings
drug_emb = torch.load(f'{NLA_DIR}/drug_embeddings.pt', map_location='cpu')
pheno_emb = torch.load(f'{NLA_DIR}/phenotype_embeddings.pt', map_location='cpu')

print(f"\ndrug_embeddings: type={type(drug_emb).__name__}")
if isinstance(drug_emb, torch.Tensor):
    print(f"  shape: {drug_emb.shape}, dtype: {drug_emb.dtype}")
elif isinstance(drug_emb, dict):
    print(f"  keys: {list(drug_emb.keys())[:5]}")

print(f"\nphenotype_embeddings: type={type(pheno_emb).__name__}")
if isinstance(pheno_emb, torch.Tensor):
    print(f"  shape: {pheno_emb.shape}, dtype: {pheno_emb.dtype}")
elif isinstance(pheno_emb, dict):
    print(f"  keys: {list(pheno_emb.keys())[:5]}")

ae_meta.json:
{
  "seed": 42,
  "hidden_dim": 256,
  "target_dim": 256,
  "fit_on": "combined_entities"
}

projection_fit_info.json:
{
  "fit_on": "combined_entities",
  "seed": 42,
  "n_drugs": 7957,
  "n_phenotypes": 3518,
  "n_total": 11475,
  "raw_dim": 768,
  "projected_dim": 256
}

drug_embeddings: type=dict
  keys: ['embeddings', 'node_indices']

phenotype_embeddings: type=dict
  keys: ['embeddings', 'node_indices']


In [ ]:
import torch

# Load embeddings
drug_data = torch.load(f'{NLA_DIR}/drug_embeddings.pt', map_location='cpu')
pheno_data = torch.load(f'{NLA_DIR}/phenotype_embeddings.pt', map_location='cpu')

drug_embs = drug_data['embeddings']
drug_indices = drug_data['node_indices']
pheno_embs = pheno_data['embeddings']
pheno_indices = pheno_data['node_indices']

print(f"Drug embeddings: {drug_embs.shape}, indices: {len(drug_indices)}")
print(f"  First 5 indices: {drug_indices[:5]}")
print(f"  Embedding norm sample: {torch.norm(drug_embs[0]).item():.4f}")

print(f"\nPhenotype embeddings: {pheno_embs.shape}, indices: {len(pheno_indices)}")
print(f"  First 5 indices: {pheno_indices[:5]}")
print(f"  Embedding norm sample: {torch.norm(pheno_embs[0]).item():.4f}")



Drug embeddings: torch.Size([7957, 256]), indices: 7957
  First 5 indices: [14012, 14013, 14014, 14015, 14016]
  Embedding norm sample: 1.0000

Phenotype embeddings: torch.Size([3518, 256]), indices: 3518
  First 5 indices: [22117, 22118, 22121, 22123, 22124]
  Embedding norm sample: 1.0000


In [ ]:
# Load fused embeddings
import torch
import numpy as np

TIER = 'hybrid'
NLA_DIR = f'{EMB_DIR}/biolinkbert/{TIER}/nonlinear_ae'

drug_data = torch.load(f'{NLA_DIR}/drug_embeddings.pt', map_location='cpu')
pheno_data = torch.load(f'{NLA_DIR}/phenotype_embeddings.pt', map_location='cpu')

drug_embs = drug_data['embeddings']           # [7957, 256]
drug_indices = drug_data['node_indices']      # PrimeKG node indices
pheno_embs = pheno_data['embeddings']         # [3518, 256]
pheno_indices = pheno_data['node_indices']

print(f"Drug embeddings: {drug_embs.shape}")
print(f"Phenotype embeddings: {pheno_embs.shape}")

Drug embeddings: torch.Size([7957, 256])
Phenotype embeddings: torch.Size([3518, 256])


In [ ]:
# Build the full node embedding tensor for R-GCN initialization
# Total nodes in PrimeKG: 129,375 (including diseases, proteins, etc.)
# We only have fused embeddings for ~11,475 nodes (drugs + phenotypes)
# Other nodes get standard random init

import torch
import torch.nn as nn

n_total_nodes = 129375  # adjust if different in your data
HIDDEN_DIM = 256

# Initialize all nodes with Xavier uniform (R-GCN default)
fused_node_emb = nn.Embedding(n_total_nodes, HIDDEN_DIM)
nn.init.xavier_uniform_(fused_node_emb.weight)

# Overwrite drug nodes with fused embeddings
for i, node_idx in enumerate(drug_indices):
    fused_node_emb.weight.data[int(node_idx)] = drug_embs[i]

# Overwrite phenotype nodes with fused embeddings
for i, node_idx in enumerate(pheno_indices):
    fused_node_emb.weight.data[int(node_idx)] = pheno_embs[i]

print(f"Initialized {n_total_nodes} node embeddings")
print(f"  Drugs initialized with fused: {len(drug_indices)}")
print(f"  Phenotypes initialized with fused: {len(pheno_indices)}")
print(f"  Other nodes (random): {n_total_nodes - len(drug_indices) - len(pheno_indices)}")

Initialized 129375 node embeddings
  Drugs initialized with fused: 7957
  Phenotypes initialized with fused: 3518
  Other nodes (random): 117900


In [ ]:
!pip install pandas numpy scipy networkx tqdm
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html


PyTorch: 2.10.0+cu128, CUDA: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.6 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 139.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 137.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from tqdm import tqdm
from torch_geometric.nn import RGCNConv
from scipy.sparse import csr_matrix
import scipy.sparse as sp


DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/"

nodes = pd.read_csv(DATA_DIR + "nodes.csv")
edges = pd.read_csv(DATA_DIR + "edges.csv")
kg    = pd.read_csv(DATA_DIR + "kg.csv", low_memory=False)

print(f"nodes: {nodes.shape}, kg: {kg.shape}")


nodes: (129375, 5), kg: (8100498, 12)


In [ ]:
INDICATION_REL = "indication"
PHENOTYPE_REL  = "disease_phenotype_positive"
DRUG_TYPE      = "drug"
SRC_COL = "x_index"
DST_COL = "y_index"
REL_COL = "relation"

# disease -> phenotypes (undirected)
phen_edges = kg[kg[REL_COL] == PHENOTYPE_REL]
disease_to_phenotypes = defaultdict(set)
for _, row in phen_edges.iterrows():
    disease_to_phenotypes[row[SRC_COL]].add(row[DST_COL])
    disease_to_phenotypes[row[DST_COL]].add(row[SRC_COL])

# drug indices
drug_indices = set(nodes[nodes["node_type"] == DRUG_TYPE]["node_index"].tolist())
drug_indices_arr = np.array(sorted(drug_indices))

# node index <-> name mappings
idx_to_name = dict(zip(nodes["node_index"], nodes["node_name"]))
name_to_idx = dict(zip(nodes["node_name"].str.lower(), nodes["node_index"]))

print(f"Diseases with phenotypes: {len(disease_to_phenotypes)}")
print(f"Drug nodes: {len(drug_indices)}")


Diseases with phenotypes: 16148
Drug nodes: 7957


In [ ]:
train_diseases = set(
    int(x.strip()) for x in open(DATA_DIR + "split/train_disease_ids.txt").readlines()
)
test_diseases = set(
    int(x.strip()) for x in open(DATA_DIR + "split/test_disease_ids.txt").readlines()
)
test_pairs  = pd.read_csv(DATA_DIR + "split/test_drug_pairs.csv")
train_pairs = pd.read_csv(DATA_DIR + "split/train_drug_pairs.csv")

# Test ground truth: disease -> set of true drugs
test_disease_to_drugs = defaultdict(set)
for _, row in test_pairs.iterrows():
    test_disease_to_drugs[int(row["disease_id"])].add(int(row["drug_id"]))

# Train ground truth (for drug degree info)
train_disease_to_drugs = defaultdict(set)
for _, row in train_pairs.iterrows():
    train_disease_to_drugs[int(row["disease_id"])].add(int(row["drug_id"]))

drug_degree = defaultdict(int)
for _, row in train_pairs.iterrows():
    drug_degree[int(row["drug_id"])] += 1

print(f"Train diseases: {len(train_diseases)}, Test diseases: {len(test_diseases)}")
print(f"Test pairs: {len(test_pairs)}")


Train diseases: 431, Test diseases: 108
Test pairs: 1568


In [ ]:
kg_train = kg[
    (~kg[SRC_COL].isin(test_diseases)) &
    (~kg[DST_COL].isin(test_diseases))
]
print(f"kg_train: {len(kg_train)} edges (masked {len(kg) - len(kg_train)})")

# PPR adjacency matrix
N = len(nodes)
src_ppr = kg_train[SRC_COL].values
dst_ppr = kg_train[DST_COL].values
rows = np.concatenate([src_ppr, dst_ppr])
cols = np.concatenate([dst_ppr, src_ppr])
data = np.ones(len(rows), dtype=np.float32)
A = csr_matrix((data, (rows, cols)), shape=(N, N))
row_sums = np.array(A.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1
D_inv = sp.diags(1.0 / row_sums)
A_norm = D_inv @ A
print(f"PPR adjacency: {A_norm.shape}, nnz={A_norm.nnz:,}")

# PyG graph for R-GCN
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_relations = sorted(kg_train["relation"].unique().tolist())
rel2id = {r: i for i, r in enumerate(all_relations)}
NUM_ORIG_RELS = len(rel2id)

src = kg_train["x_index"].values
dst = kg_train["y_index"].values
rel = np.array([rel2id[r] for r in kg_train["relation"].values])

edge_src = np.concatenate([src, dst])
edge_dst = np.concatenate([dst, src])
edge_rel = np.concatenate([rel, rel + NUM_ORIG_RELS])

NUM_RELATIONS = NUM_ORIG_RELS * 2
NUM_NODES = len(nodes)

edge_index = torch.tensor(np.stack([edge_src, edge_dst]), dtype=torch.long).to(DEVICE)
edge_type  = torch.tensor(edge_rel, dtype=torch.long).to(DEVICE)

print(f"Nodes: {NUM_NODES:,}, Edges: {edge_index.shape[1]:,}, Relations: {NUM_RELATIONS}")
print(f"Device: {DEVICE}")


kg_train: 8081958 edges (masked 18540)
PPR adjacency: (129375, 129375), nnz=8,080,782
Nodes: 129,375, Edges: 16,163,916, Relations: 60
Device: cuda


In [ ]:
HIDDEN_DIM = 256
NUM_BASES  = 15
NUM_LAYERS = 3
NUM_HEADS  = 4
DROPOUT    = 0.2

class DrugConditionedCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.score_proj = nn.Linear(dim, 1)
    def forward(self, drug_emb, pheno_embs, pheno_mask=None):
        query = drug_emb.unsqueeze(1)
        attn_out, _ = self.attn(query, pheno_embs, pheno_embs, key_padding_mask=pheno_mask)
        return self.score_proj(attn_out.squeeze(1)).squeeze(-1)

class PhenoDrugModel(nn.Module):
    def __init__(self, num_nodes, num_relations, hidden_dim,
                 num_bases=10, num_layers=2, num_heads=4, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(RGCNConv(hidden_dim, hidden_dim,
                                       num_relations=num_relations, num_bases=num_bases))
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.cross_attn = DrugConditionedCrossAttention(hidden_dim, num_heads, dropout)
        self.dropout = dropout
    def encode(self, edge_index, edge_type):
        x = self.node_emb.weight
        for conv, norm in zip(self.convs, self.norms):
            residual = x  # ADD THIS LINE
            x = conv(x, edge_index, edge_type)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual  # ADD THIS LINE
        return x
    def score(self, node_embs, drug_indices, pheno_indices_list, pheno_mask):
        return self.cross_attn(node_embs[drug_indices], node_embs[pheno_indices_list], pheno_mask)

model = PhenoDrugModel(
    NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT
).to(DEVICE)

model.load_state_dict(torch.load(DATA_DIR + "best_model.pt", map_location=DEVICE))
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")


Model loaded: 36,534,157 parameters


In [ ]:
# === FEATURE-LEVEL FUSION: inject fused embeddings, then fine-tune ===
import torch
import torch.nn as nn

TIER = 'hybrid'
NLA_DIR = f'{EMB_DIR}/biolinkbert/{TIER}/nonlinear_ae'

drug_data = torch.load(f'{NLA_DIR}/drug_embeddings.pt', map_location='cpu')
pheno_data = torch.load(f'{NLA_DIR}/phenotype_embeddings.pt', map_location='cpu')

# Create a fresh model
fused_model = PhenoDrugModel(
    NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT
).to(DEVICE)

# Load existing checkpoint
fused_model.load_state_dict(torch.load(DATA_DIR + "best_model.pt", map_location=DEVICE))

# Now overwrite drug + phenotype node embeddings with fused
with torch.no_grad():
    for i, node_idx in enumerate(drug_data['node_indices']):
        fused_model.node_emb.weight.data[int(node_idx)] = drug_data['embeddings'][i].to(DEVICE)
    for i, node_idx in enumerate(pheno_data['node_indices']):
        fused_model.node_emb.weight.data[int(node_idx)] = pheno_data['embeddings'][i].to(DEVICE)

print(f"Fused embeddings injected:")
print(f"  Drugs: {len(drug_data['node_indices'])}")
print(f"  Phenotypes: {len(pheno_data['node_indices'])}")
print(f"  Other nodes (kept from checkpoint): {NUM_NODES - len(drug_data['node_indices']) - len(pheno_data['node_indices'])}")

Fused embeddings injected:
  Drugs: 7957
  Phenotypes: 3518
  Other nodes (kept from checkpoint): 117900


In [ ]:
def recall_at_k(ranked_drugs, true_drugs, k):
    top_k = set(ranked_drugs[:k])
    hits = len(top_k & set(true_drugs))
    return hits / len(true_drugs) if true_drugs else 0.0

def reciprocal_rank(ranked_drugs, true_drugs):
    true_set = set(true_drugs)
    for rank, d in enumerate(ranked_drugs, start=1):
        if d in true_set:
            return 1.0 / rank
    return 0.0

def ppr_scores(seed_indices, A_norm, alpha=0.15, max_iter=50, tol=1e-6):
    N = A_norm.shape[0]
    s = np.zeros(N, dtype=np.float32)
    if len(seed_indices) == 0:
        return s
    s[seed_indices] = 1.0 / len(seed_indices)
    r = s.copy()
    A_T = A_norm.T
    for _ in range(max_iter):
        r_new = (1 - alpha) * A_T.dot(r) + alpha * s
        if np.linalg.norm(r_new - r, 1) < tol:
            break
        r = r_new
    return r_new

def pad_pheno_batch(pheno_lists, device):
    max_len = max(len(p) for p in pheno_lists)
    batch_size = len(pheno_lists)
    padded = torch.zeros(batch_size, max_len, dtype=torch.long, device=device)
    mask   = torch.ones(batch_size, max_len, dtype=torch.bool, device=device)
    for i, p in enumerate(pheno_lists):
        padded[i, :len(p)] = torch.tensor(p, dtype=torch.long)
        mask[i, :len(p)]   = False
    return padded, mask

print("Metrics and utilities defined.")


Metrics and utilities defined.


In [ ]:
# === Fine-tune fused model with margin ranking loss ===
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from collections import defaultdict
import random

# Hyperparameters from her midterm
LR = 1e-3
WEIGHT_DECAY = 1e-5
MARGIN = 1.0
EPOCHS = 30  # fine-tuning, fewer than full training
BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 4
GRAD_CLIP = 1.0

# Build training pairs from train_disease_to_drugs
# Each pair: (disease_idx, true_drug_idx) with phenotypes
training_pairs = []
for d_idx, drugs in train_disease_to_drugs.items():
    phenos = list(disease_to_phenotypes.get(d_idx, []))
    if not phenos:
        continue
    for drug in drugs:
        training_pairs.append((d_idx, drug, phenos))

print(f"Training pairs: {len(training_pairs)}")

# Degree-weighted negative sampling: drugs with more indications sampled MORE often as negatives
drug_degree = defaultdict(int)
for d_idx, drugs in train_disease_to_drugs.items():
    for drug in drugs:
        drug_degree[drug] += 1

drug_indices_list = list(drug_indices_arr)
drug_weights = np.array([drug_degree.get(d, 1) for d in drug_indices_list], dtype=np.float64)
drug_weights = drug_weights / drug_weights.sum()

Training pairs: 2703


In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random

In [ ]:
# === FULL TRAIN: feature-level fusion from scratch ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import time
from collections import defaultdict

# Hyperparameters - matching her midterm spec
LR = 1e-3
WEIGHT_DECAY = 1e-5
MARGIN = 1.0
EPOCHS = 80  # full training, more epochs
BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 4
GRAD_CLIP = 1.0
EVAL_EVERY = 5

# Path setup
EMB_DIR = '/content/drive/MyDrive/Colab Notebooks/embeddings_256'  # adjust your path
TIER = 'hybrid'
NLA_DIR = f'{EMB_DIR}/biolinkbert/{TIER}/nonlinear_ae'

# === Step 1: Fresh model with fused embeddings as initialization ===
fused_model = PhenoDrugModel(
    NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT
).to(DEVICE)

# Don't load best_model.pt - train from scratch with fused init
# Just inject fused embeddings into freshly initialized model

drug_data = torch.load(f'{NLA_DIR}/drug_embeddings.pt', map_location='cpu')
pheno_data = torch.load(f'{NLA_DIR}/phenotype_embeddings.pt', map_location='cpu')

with torch.no_grad():
    for i, node_idx in enumerate(drug_data['node_indices']):
        fused_model.node_emb.weight.data[int(node_idx)] = drug_data['embeddings'][i].to(DEVICE)
    for i, node_idx in enumerate(pheno_data['node_indices']):
        fused_model.node_emb.weight.data[int(node_idx)] = pheno_data['embeddings'][i].to(DEVICE)

print(f"Fresh model initialized with fused embeddings")
print(f"  Drugs: {len(drug_data['node_indices'])}")
print(f"  Phenotypes: {len(pheno_data['node_indices'])}")

# === Step 2: Build training pairs ===
training_pairs = []
for d_idx, drugs in train_disease_to_drugs.items():
    phenos = list(disease_to_phenotypes.get(d_idx, []))
    if not phenos:
        continue
    for drug in drugs:
        training_pairs.append((d_idx, drug, phenos))

print(f"Training pairs: {len(training_pairs)}")

# Degree-weighted negative sampling
drug_degree = defaultdict(int)
for d_idx, drugs in train_disease_to_drugs.items():
    for drug in drugs:
        drug_degree[drug] += 1

drug_indices_list = list(drug_indices_arr)
drug_weights = np.array([drug_degree.get(d, 1) for d in drug_indices_list], dtype=np.float64)
drug_weights = drug_weights / drug_weights.sum()

Fresh model initialized with fused embeddings
  Drugs: 7957
  Phenotypes: 3518
Training pairs: 2703


In [ ]:
# Stop current training, then:
SCALE = 5.0  # or 10.0

# Re-init fresh model
fused_model = PhenoDrugModel(NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT).to(DEVICE)

with torch.no_grad():
    for i, node_idx in enumerate(drug_data['node_indices']):
        fused_model.node_emb.weight.data[int(node_idx)] = drug_data['embeddings'][i].to(DEVICE) * SCALE
    for i, node_idx in enumerate(pheno_data['node_indices']):
        fused_model.node_emb.weight.data[int(node_idx)] = pheno_data['embeddings'][i].to(DEVICE) * SCALE

# Then run training cell again with this fresh model

In [ ]:
# === Step 3: Train from scratch ===
optimizer = optim.AdamW(fused_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

best_mrr = 0.0
fused_model.train()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    random.shuffle(training_pairs)
    epoch_loss = 0.0
    n_batches = 0

    optimizer.zero_grad()

    for batch_start in range(0, len(training_pairs), BATCH_SIZE):
        batch = training_pairs[batch_start:batch_start + BATCH_SIZE]
        if not batch:
            continue

        node_embs = fused_model.encode(edge_index, edge_type)

        pos_drugs = [pair[1] for pair in batch]
        neg_drugs = np.random.choice(drug_indices_list, size=len(batch), p=drug_weights).tolist()
        for i, (d_idx, pos_drug, _) in enumerate(batch):
            true_set = train_disease_to_drugs.get(d_idx, set())
            attempts = 0
            while neg_drugs[i] in true_set and attempts < 5:
                neg_drugs[i] = np.random.choice(drug_indices_list, p=drug_weights)
                attempts += 1

        pheno_lists = [pair[2] for pair in batch]
        pheno_padded, pheno_mask = pad_pheno_batch(pheno_lists, DEVICE)

        pos_drugs_t = torch.tensor(pos_drugs, dtype=torch.long, device=DEVICE)
        neg_drugs_t = torch.tensor(neg_drugs, dtype=torch.long, device=DEVICE)

        pos_scores = fused_model.score(node_embs, pos_drugs_t, pheno_padded, pheno_mask)
        neg_scores = fused_model.score(node_embs, neg_drugs_t, pheno_padded, pheno_mask)

        loss = F.margin_ranking_loss(
            pos_scores, neg_scores,
            torch.ones_like(pos_scores),
            margin=MARGIN
        )
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        n_batches += 1
        epoch_loss += loss.item() * GRAD_ACCUM_STEPS

        if n_batches % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(fused_model.parameters(), GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad()

    if n_batches % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(fused_model.parameters(), GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad()

    epoch_time = time.time() - epoch_start

    # Evaluate every 5 epochs
    if (epoch + 1) % EVAL_EVERY == 0:
        fused_model.eval()
        mrrs = []
        with torch.no_grad():
            ne = fused_model.encode(edge_index, edge_type)
            for d in test_diseases:
                ph = list(disease_to_phenotypes.get(d, []))
                td = list(test_disease_to_drugs.get(d, []))
                if not ph or not td: continue
                ad = torch.tensor(drug_indices_arr, dtype=torch.long, device=DEVICE)
                sc = []
                for cs in range(0, len(drug_indices_arr), 512):
                    ce = min(cs+512, len(drug_indices_arr))
                    cp, cm = pad_pheno_batch([ph]*(ce-cs), DEVICE)
                    sc.append(fused_model.score(ne, ad[cs:ce], cp, cm).cpu().numpy())
                sc = np.concatenate(sc)
                rd = drug_indices_arr[np.argsort(-sc)].tolist()
                mrrs.append(reciprocal_rank(rd, td))

        eval_mrr = np.mean(mrrs)
        print(f"Epoch {epoch+1} ({epoch_time:.0f}s): loss={epoch_loss/n_batches:.4f}, eval MRR={eval_mrr:.4f}")

        if eval_mrr > best_mrr:
            best_mrr = eval_mrr
            torch.save(fused_model.state_dict(), DATA_DIR + "best_model_fused_full.pt")
            print(f"  → saved (new best)")

        scheduler.step(eval_mrr)
        fused_model.train()
    else:
        print(f"Epoch {epoch+1} ({epoch_time:.0f}s): loss={epoch_loss/n_batches:.4f}")

print(f"\nFinal best MRR: {best_mrr:.4f}")

Epoch 1 (50s): loss=0.5595
Epoch 2 (50s): loss=0.4100
Epoch 3 (50s): loss=0.3831
Epoch 4 (50s): loss=0.3623
Epoch 5 (50s): loss=0.3589, eval MRR=0.0240
  → saved (new best)
Epoch 6 (50s): loss=0.3475
Epoch 7 (50s): loss=0.3411
Epoch 8 (50s): loss=0.3273
Epoch 9 (50s): loss=0.3495
Epoch 10 (50s): loss=0.3332, eval MRR=0.0283
  → saved (new best)
Epoch 11 (50s): loss=0.3322


KeyboardInterrupt: 

In [ ]:
# Load Beatrice's hybrid descriptions
import json
import os

# Path to descriptions folder (adjust if different)
DESC_DIR = '/content/drive/MyDrive/Colab Notebooks/descriptions'  # adjust!

# List what's available
for f in sorted(os.listdir(DESC_DIR)):
    print(f)

drugs_gpt4o.json
drugs_hybrid.json
drugs_tier1.json
drugs_tier2.json
phenotypes_gpt4o.json
phenotypes_hybrid.json
phenotypes_tier1.json
phenotypes_tier2.json


In [ ]:
# Load and inspect
with open(f'{DESC_DIR}/drugs_hybrid.json') as f:
    drug_descs = json.load(f)
with open(f'{DESC_DIR}/phenotypes_hybrid.json') as f:
    pheno_descs = json.load(f)

print(f"Drugs: {len(drug_descs)}")
print(f"Phenotypes: {len(pheno_descs)}")

# Inspect structure
print(f"\nDrug type: {type(drug_descs).__name__}")
if isinstance(drug_descs, dict):
    sample_key = list(drug_descs.keys())[0]
    print(f"Sample key: {sample_key} (type: {type(sample_key).__name__})")
    print(f"Sample value type: {type(drug_descs[sample_key]).__name__}")
    print(f"Sample value: {str(drug_descs[sample_key])[:500]}")
elif isinstance(drug_descs, list):
    print(f"First entry: {str(drug_descs[0])[:500]}")

Drugs: 7957
Phenotypes: 3518

Drug type: dict
Sample key: 14012 (type: str)
Sample value type: dict
Sample value: {'name': 'Copper', 'text': "Copper. Targets: A1BG, A2M, ACTG1, ACTN1, ACY1, AFM, AGT, AHCY, AHSG, AKR1A1. Copper is an essential trace element that plays a critical role in various physiological processes, including acting as a cofactor for enzymes involved in redox reactions, iron metabolism, and connective tissue formation. Its mechanism of action involves facilitating electron transfer in enzymes such as cytochrome c oxidase and superoxide dismutase. Clinically, copper is used in the treatmen


In [ ]:
# Step 1: Encode descriptions with BioLinkBERT (raw 768-dim, no autoencoder)
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm

device = DEVICE
tokenizer = AutoTokenizer.from_pretrained('michiyasunaga/BioLinkBERT-base')
blb = AutoModel.from_pretrained('michiyasunaga/BioLinkBERT-base').to(device).eval()

def encode_batch(texts, batch_size=32, max_length=256):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, truncation=True,
                          max_length=max_length, return_tensors='pt').to(device)
        with torch.no_grad():
            out = blb(**tokens)
        mask = tokens['attention_mask'].unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        embs.append(pooled.cpu())
    return torch.cat(embs)

# Build sorted lists (by node_index) of texts
drug_items = sorted(drug_descs.items(), key=lambda x: int(x[0]))
drug_indices_raw = [int(k) for k, _ in drug_items]
drug_texts = [v['text'] for _, v in drug_items]

pheno_items = sorted(pheno_descs.items(), key=lambda x: int(x[0]))
pheno_indices_raw = [int(k) for k, _ in pheno_items]
pheno_texts = [v['text'] for _, v in pheno_items]

print(f"Encoding {len(drug_texts)} drugs...")
drug_emb_raw = encode_batch(drug_texts)
print(f"Drug embeddings: {drug_emb_raw.shape}")  # [7957, 768]

print(f"\nEncoding {len(pheno_texts)} phenotypes...")
pheno_emb_raw = encode_batch(pheno_texts)
print(f"Phenotype embeddings: {pheno_emb_raw.shape}")  # [3518, 768]

# Save to avoid re-encoding
torch.save({
    'drug_emb': drug_emb_raw, 'drug_indices': drug_indices_raw,
    'pheno_emb': pheno_emb_raw, 'pheno_indices': pheno_indices_raw
}, DATA_DIR + 'biolinkbert_raw_hybrid.pt')
print("\nSaved")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Encoding: 100%|██████████| 249/249 [00:20<00:00, 12.31it/s]


Drug embeddings: torch.Size([7957, 768])

Encoding 3518 phenotypes...


Encoding: 100%|██████████| 110/110 [00:09<00:00, 11.30it/s]


Phenotype embeddings: torch.Size([3518, 768])

Saved


In [ ]:
# Step 2: Project 768 → 256 with random init, inject into fresh model
import torch.nn as nn

projection = nn.Linear(768, 256).to(DEVICE)
nn.init.xavier_uniform_(projection.weight)

with torch.no_grad():
    drug_emb_proj = projection(drug_emb_raw.to(DEVICE))
    pheno_emb_proj = projection(pheno_emb_raw.to(DEVICE))

print(f"Projected drug magnitudes: mean={drug_emb_proj.norm(dim=1).mean():.3f}")
print(f"Projected pheno magnitudes: mean={pheno_emb_proj.norm(dim=1).mean():.3f}")

# Init fresh model
fused_model = PhenoDrugModel(NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT).to(DEVICE)

with torch.no_grad():
    for i, idx in enumerate(drug_indices_raw):
        fused_model.node_emb.weight.data[idx] = drug_emb_proj[i]
    for i, idx in enumerate(pheno_indices_raw):
        fused_model.node_emb.weight.data[idx] = pheno_emb_proj[i]

print("Fresh model initialized with raw BioLinkBERT (projected)")

Projected drug magnitudes: mean=5.753
Projected pheno magnitudes: mean=5.932
Fresh model initialized with raw BioLinkBERT (projected)


In [ ]:
# Step 3: Train (same loop as before)
import torch.optim as optim
import torch.nn.functional as F
import random
import time

EPOCHS = 30
optimizer = optim.AdamW(fused_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_mrr = 0.0
fused_model.train()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    random.shuffle(training_pairs)
    epoch_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    for batch_start in range(0, len(training_pairs), BATCH_SIZE):
        batch = training_pairs[batch_start:batch_start + BATCH_SIZE]
        if not batch: continue

        node_embs = fused_model.encode(edge_index, edge_type)

        pos_drugs = [pair[1] for pair in batch]
        neg_drugs = np.random.choice(drug_indices_list, size=len(batch), p=drug_weights).tolist()
        for i, (d_idx, _, _) in enumerate(batch):
            true_set = train_disease_to_drugs.get(d_idx, set())
            attempts = 0
            while neg_drugs[i] in true_set and attempts < 5:
                neg_drugs[i] = np.random.choice(drug_indices_list, p=drug_weights)
                attempts += 1

        pheno_lists = [pair[2] for pair in batch]
        pheno_padded, pheno_mask = pad_pheno_batch(pheno_lists, DEVICE)

        pos_drugs_t = torch.tensor(pos_drugs, dtype=torch.long, device=DEVICE)
        neg_drugs_t = torch.tensor(neg_drugs, dtype=torch.long, device=DEVICE)

        pos_scores = fused_model.score(node_embs, pos_drugs_t, pheno_padded, pheno_mask)
        neg_scores = fused_model.score(node_embs, neg_drugs_t, pheno_padded, pheno_mask)

        loss = F.margin_ranking_loss(pos_scores, neg_scores, torch.ones_like(pos_scores), margin=MARGIN)
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()
        n_batches += 1
        epoch_loss += loss.item() * GRAD_ACCUM_STEPS

        if n_batches % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(fused_model.parameters(), GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad()

    if n_batches % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(fused_model.parameters(), GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad()

    epoch_time = time.time() - epoch_start

    # Eval every 3 epochs
    if (epoch + 1) % 3 == 0:
        fused_model.eval()
        mrrs = []
        with torch.no_grad():
            ne = fused_model.encode(edge_index, edge_type)
            for d in test_diseases:
                ph = list(disease_to_phenotypes.get(d, []))
                td = list(test_disease_to_drugs.get(d, []))
                if not ph or not td: continue
                ad = torch.tensor(drug_indices_arr, dtype=torch.long, device=DEVICE)
                sc = []
                for cs in range(0, len(drug_indices_arr), 512):
                    ce = min(cs+512, len(drug_indices_arr))
                    cp, cm = pad_pheno_batch([ph]*(ce-cs), DEVICE)
                    sc.append(fused_model.score(ne, ad[cs:ce], cp, cm).cpu().numpy())
                sc = np.concatenate(sc)
                rd = drug_indices_arr[np.argsort(-sc)].tolist()
                mrrs.append(reciprocal_rank(rd, td))
        eval_mrr = np.mean(mrrs)
        print(f"Epoch {epoch+1} ({epoch_time:.0f}s): loss={epoch_loss/n_batches:.4f}, eval MRR={eval_mrr:.4f}")
        if eval_mrr > best_mrr:
            best_mrr = eval_mrr
            torch.save(fused_model.state_dict(), DATA_DIR + "best_model_fused_raw.pt")
            print(f"  → saved (new best)")
        fused_model.train()
    else:
        print(f"Epoch {epoch+1} ({epoch_time:.0f}s): loss={epoch_loss/n_batches:.4f}")

print(f"\nFinal best MRR: {best_mrr:.4f}")

Epoch 1 (50s): loss=0.5834
Epoch 2 (50s): loss=0.3917
Epoch 3 (50s): loss=0.3953, eval MRR=0.0060
  → saved (new best)
Epoch 4 (50s): loss=0.3800
Epoch 5 (50s): loss=0.3413
Epoch 6 (50s): loss=0.3282, eval MRR=0.0153
  → saved (new best)
Epoch 7 (50s): loss=0.3461
Epoch 8 (50s): loss=0.3555
Epoch 9 (50s): loss=0.3473, eval MRR=0.0084
Epoch 10 (50s): loss=0.3290
Epoch 11 (50s): loss=0.3406
Epoch 12 (50s): loss=0.3161, eval MRR=0.0066
Epoch 13 (50s): loss=0.3073
Epoch 14 (50s): loss=0.3186
Epoch 15 (50s): loss=0.3150, eval MRR=0.0183
  → saved (new best)
Epoch 16 (50s): loss=0.3222
Epoch 17 (50s): loss=0.3067
Epoch 18 (50s): loss=0.3079, eval MRR=0.0123
Epoch 19 (50s): loss=0.2953
Epoch 20 (50s): loss=0.2975
Epoch 21 (50s): loss=0.3073, eval MRR=0.0571
  → saved (new best)
Epoch 22 (50s): loss=0.2695
Epoch 23 (50s): loss=0.2584
Epoch 24 (50s): loss=0.2599, eval MRR=0.0434


KeyboardInterrupt: 

In [ ]:
# === Approach A: Concatenated input embedding ===
# Modify model so node_emb has BOTH a learnable random part AND fixed text features
# Each layer sees both signals throughout the network

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv

# Load raw BioLinkBERT (768-dim) embeddings we encoded earlier
# If you don't have them in memory, reload:
saved = torch.load(DATA_DIR + 'biolinkbert_raw_hybrid.pt')
drug_emb_raw = saved['drug_emb']
drug_indices_raw = saved['drug_indices']
pheno_emb_raw = saved['pheno_emb']
pheno_indices_raw = saved['pheno_indices']

# Build full text feature tensor for ALL nodes (zero for non-drug/non-pheno)
TEXT_DIM = 768
text_features = torch.zeros(NUM_NODES, TEXT_DIM)
for i, idx in enumerate(drug_indices_raw):
    text_features[idx] = drug_emb_raw[i]
for i, idx in enumerate(pheno_indices_raw):
    text_features[idx] = pheno_emb_raw[i]

text_features = text_features.to(DEVICE)
print(f"Text features tensor: {text_features.shape}")
print(f"Coverage: {(text_features.norm(dim=1) > 0).sum().item()}/{NUM_NODES} nodes")

Text features tensor: torch.Size([129375, 768])
Coverage: 11475/129375 nodes


In [ ]:
# Define new model class with concatenated input
class FusedPhenoDrugModel(nn.Module):
    def __init__(self, num_nodes, num_relations, hidden_dim, text_features,
                 num_bases=10, num_layers=2, num_heads=4, dropout=0.2, text_dim=768):
        super().__init__()
        # Learnable random embedding (smaller than before to keep total dim manageable)
        self.random_emb = nn.Embedding(num_nodes, hidden_dim)
        nn.init.xavier_uniform_(self.random_emb.weight)

        # Fixed text features (registered as buffer, not trainable)
        self.register_buffer('text_features', text_features)

        # Project text 768 → hidden_dim, trainable
        self.text_projection = nn.Linear(text_dim, hidden_dim)
        nn.init.xavier_uniform_(self.text_projection.weight)

        # R-GCN layers operate on 2*hidden_dim (concat of random + projected text)
        combined_dim = hidden_dim * 2
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(RGCNConv(combined_dim, combined_dim,
                                       num_relations=num_relations, num_bases=num_bases))
            self.norms.append(nn.LayerNorm(combined_dim))

        # Cross-attention works on combined dim too
        self.attn = nn.MultiheadAttention(combined_dim, num_heads, dropout=dropout, batch_first=True)
        self.score_proj = nn.Linear(combined_dim, 1)
        self.dropout = dropout

    def encode(self, edge_index, edge_type):
        random_part = self.random_emb.weight  # [num_nodes, hidden_dim]
        text_part = self.text_projection(self.text_features)  # [num_nodes, hidden_dim]
        x = torch.cat([random_part, text_part], dim=-1)  # [num_nodes, 2*hidden_dim]

        for conv, norm in zip(self.convs, self.norms):
            residual = x
            x = conv(x, edge_index, edge_type)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual
        return x

    def score(self, node_embs, drug_indices, pheno_indices_list, pheno_mask):
        drug_q = node_embs[drug_indices].unsqueeze(1)
        pheno_kv = node_embs[pheno_indices_list]
        attn_out, _ = self.attn(drug_q, pheno_kv, pheno_kv, key_padding_mask=pheno_mask)
        return self.score_proj(attn_out.squeeze(1)).squeeze(-1)


# Instantiate
fused_model = FusedPhenoDrugModel(
    NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, text_features,
    NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT, text_dim=768
).to(DEVICE)

print(f"Fused model created with concatenated input")
print(f"  Total params: {sum(p.numel() for p in fused_model.parameters()):,}")
print(f"  Hidden dim: {HIDDEN_DIM} (random) + {HIDDEN_DIM} (text projected) = {HIDDEN_DIM*2}")

Fused model created with concatenated input
  Total params: 46,958,221
  Hidden dim: 256 (random) + 256 (text projected) = 512


In [36]:
# Train (same loop, just uses new fused_model)
import torch.optim as optim
import random
import time
import numpy as np

EPOCHS = 30
optimizer = optim.AdamW(fused_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_mrr = 0.0
fused_model.train()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    random.shuffle(training_pairs)
    epoch_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    for batch_start in range(0, len(training_pairs), BATCH_SIZE):
        batch = training_pairs[batch_start:batch_start + BATCH_SIZE]
        if not batch: continue

        node_embs = fused_model.encode(edge_index, edge_type)

        pos_drugs = [pair[1] for pair in batch]
        neg_drugs = np.random.choice(drug_indices_list, size=len(batch), p=drug_weights).tolist()
        for i, (d_idx, _, _) in enumerate(batch):
            true_set = train_disease_to_drugs.get(d_idx, set())
            attempts = 0
            while neg_drugs[i] in true_set and attempts < 5:
                neg_drugs[i] = np.random.choice(drug_indices_list, p=drug_weights)
                attempts += 1

        pheno_lists = [pair[2] for pair in batch]
        pheno_padded, pheno_mask = pad_pheno_batch(pheno_lists, DEVICE)

        pos_drugs_t = torch.tensor(pos_drugs, dtype=torch.long, device=DEVICE)
        neg_drugs_t = torch.tensor(neg_drugs, dtype=torch.long, device=DEVICE)

        pos_scores = fused_model.score(node_embs, pos_drugs_t, pheno_padded, pheno_mask)
        neg_scores = fused_model.score(node_embs, neg_drugs_t, pheno_padded, pheno_mask)

        loss = F.margin_ranking_loss(pos_scores, neg_scores, torch.ones_like(pos_scores), margin=MARGIN)
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()
        n_batches += 1
        epoch_loss += loss.item() * GRAD_ACCUM_STEPS

        if n_batches % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(fused_model.parameters(), GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad()

    if n_batches % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(fused_model.parameters(), GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad()

    epoch_time = time.time() - epoch_start

    if (epoch + 1) % 3 == 0:
        fused_model.eval()
        mrrs = []
        with torch.no_grad():
            ne = fused_model.encode(edge_index, edge_type)
            for d in test_diseases:
                ph = list(disease_to_phenotypes.get(d, []))
                td = list(test_disease_to_drugs.get(d, []))
                if not ph or not td: continue
                ad = torch.tensor(drug_indices_arr, dtype=torch.long, device=DEVICE)
                sc = []
                for cs in range(0, len(drug_indices_arr), 512):
                    ce = min(cs+512, len(drug_indices_arr))
                    cp, cm = pad_pheno_batch([ph]*(ce-cs), DEVICE)
                    sc.append(fused_model.score(ne, ad[cs:ce], cp, cm).cpu().numpy())
                sc = np.concatenate(sc)
                rd = drug_indices_arr[np.argsort(-sc)].tolist()
                mrrs.append(reciprocal_rank(rd, td))
        eval_mrr = np.mean(mrrs)
        print(f"Epoch {epoch+1} ({epoch_time:.0f}s): loss={epoch_loss/n_batches:.4f}, eval MRR={eval_mrr:.4f}")
        if eval_mrr > best_mrr:
            best_mrr = eval_mrr
            torch.save(fused_model.state_dict(), DATA_DIR + "best_model_concat_fusion.pt")
            print(f"  → saved (new best)")
        fused_model.train()
    else:
        print(f"Epoch {epoch+1} ({epoch_time:.0f}s): loss={epoch_loss/n_batches:.4f}")

print(f"\nFinal best MRR: {best_mrr:.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.90 GiB. GPU 0 has a total capacity of 79.25 GiB of which 298.81 MiB is free. Including non-PyTorch memory, this process has 78.95 GiB memory in use. Of the allocated memory 44.74 GiB is allocated by PyTorch, and 33.70 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [34]:
# Reduce dimensions to fit memory
# Total combined dim stays HIDDEN_DIM (256), split as 128 + 128

class FusedPhenoDrugModel(nn.Module):
    def __init__(self, num_nodes, num_relations, hidden_dim, text_features,
                 num_bases=10, num_layers=2, num_heads=4, dropout=0.2, text_dim=768):
        super().__init__()
        # Split hidden_dim into two halves
        self.half_dim = hidden_dim // 2  # 128

        # Random learnable embedding (smaller now)
        self.random_emb = nn.Embedding(num_nodes, self.half_dim)
        nn.init.xavier_uniform_(self.random_emb.weight)

        # Fixed text features
        self.register_buffer('text_features', text_features)

        # Project text 768 → half_dim
        self.text_projection = nn.Linear(text_dim, self.half_dim)
        nn.init.xavier_uniform_(self.text_projection.weight)

        # R-GCN operates on hidden_dim (256) total = random (128) + text projected (128)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(RGCNConv(hidden_dim, hidden_dim,
                                       num_relations=num_relations, num_bases=num_bases))
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.score_proj = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def encode(self, edge_index, edge_type):
        random_part = self.random_emb.weight  # [num_nodes, 128]
        text_part = self.text_projection(self.text_features)  # [num_nodes, 128]
        x = torch.cat([random_part, text_part], dim=-1)  # [num_nodes, 256]

        for conv, norm in zip(self.convs, self.norms):
            residual = x
            x = conv(x, edge_index, edge_type)
            x = norm(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + residual
        return x

    def score(self, node_embs, drug_indices, pheno_indices_list, pheno_mask):
        drug_q = node_embs[drug_indices].unsqueeze(1)
        pheno_kv = node_embs[pheno_indices_list]
        attn_out, _ = self.attn(drug_q, pheno_kv, pheno_kv, key_padding_mask=pheno_mask)
        return self.score_proj(attn_out.squeeze(1)).squeeze(-1)


# Clear old GPU memory first
import gc
del fused_model
gc.collect()
torch.cuda.empty_cache()

# Instantiate with same total HIDDEN_DIM as original model (256)
fused_model = FusedPhenoDrugModel(
    NUM_NODES, NUM_RELATIONS, HIDDEN_DIM, text_features,
    NUM_BASES, NUM_LAYERS, NUM_HEADS, DROPOUT, text_dim=768
).to(DEVICE)

print(f"Fused model created with concat (random {fused_model.half_dim} + text {fused_model.half_dim} = {HIDDEN_DIM})")
print(f"  Total params: {sum(p.numel() for p in fused_model.parameters()):,}")

Fused model created with concat (random 128 + text 128 = 256)
  Total params: 20,072,589
